# PathSnap — Dataset Build Pipeline (Kaggle / Colab)

Turns the raw per-trip folders (`.nodes` / `.arcs` / `.track` / `.route`) into two
flat, **route-blind feature tables** with **estimated point-level training labels**:

- `outputs/train_features.parquet` — 80 training trips
- `outputs/student_test_features.parquet` — 10 student-test trips
- `outputs/*_diagnostics.csv` — per-trip QA (candidate coverage, alignment quality)
- `outputs/*_alignment_diagnostics.parquet` — reference alignment estimates, stored separately from inputs
- `outputs/feature_scaler.joblib` — impute+scale pipeline, **fit on labeled train rows only**

Run cells top to bottom. Attach the raw dataset to Kaggle (or upload/extract it
on Colab), and adjust `DATA_ROOT` in the configuration cell only if autodetection
fails. Candidate generation, all 15 features, and preprocessing structure are
unchanged from the original notebook.

**Important:** The ordered `.route` lacks GPS-point timestamps. The point-level
labels here come from monotonic alignment and are *estimates*, not independently
verified observation-level ground truth. Inspect alignment diagnostics and
real-world map overlays before training. The new alignment step has not been
benchmarked on the full 80+10-trip dataset.

## 0. Get the dataset onto this machine

Pick whichever matches where you're running this:

**Kaggle:** click **"+ Add Data"** (right sidebar) and attach your `Student folder`
dataset. It will appear under `/kaggle/input/<your-dataset-slug>/...` — no need to
unzip anything yourself, Kaggle datasets are already extracted.

**Colab — upload directly:** run this in a new cell, then select your zip file:
```python
from google.colab import files
uploaded = files.upload()   # pick Student_folder.zip
```

**Colab — from Google Drive:**
```python
from google.colab import drive
drive.mount('/content/drive')
# then make sure DATA_ROOT below points at the extracted "Student folder" in Drive
```

The next cell auto-extracts any `*.zip` it finds whose name contains "student" or
"pathsnap", and auto-detects the resulting data folder. If your dataset is already
extracted (e.g. a Kaggle dataset), it does nothing and just finds it.


In [ ]:
# Quiet, failure-tolerant installs — these packages are already present on
# standard Kaggle/Colab images, so this is just a safety net; it won't halt
# the notebook if pip is restricted or everything is already installed.
import subprocess
import sys


def _pip_install(*packages):
    try:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)
    except subprocess.CalledProcessError:
        print("Install skipped (already present or restricted) — continuing.")


_pip_install("pyarrow", "scikit-learn", "joblib", "scipy", "pandas", "numpy")


In [ ]:
import os
import sys
import time
import zipfile
from dataclasses import dataclass
from pathlib import Path
from typing import Optional

import joblib
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = Path("/kaggle/input").exists()
print(f"Colab: {IN_COLAB} | Kaggle: {IN_KAGGLE}")


def ensure_data_extracted():
    """Auto-extract any uploaded Student*/PathSnap*.zip. No-op if the dataset
    is already extracted (e.g. a Kaggle dataset, or Drive-mounted folders)."""
    search_bases = [Path("/content"), Path("."), Path("/kaggle/working")]
    out_root = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
    for base in search_bases:
        if not base.exists():
            continue
        for zpath in list(base.glob("*.zip")) + list(base.glob("*/*.zip")):
            name_lower = zpath.stem.lower()
            if "student" not in name_lower and "pathsnap" not in name_lower:
                continue
            out_dir = out_root / zpath.stem
            if out_dir.exists():
                continue
            print(f"Extracting {zpath} -> {out_dir}")
            with zipfile.ZipFile(zpath) as zf:
                zf.extractall(out_dir)


ensure_data_extracted()


## 1. Config — paths, candidate count, feature columns

`find_data_root` searches `/content`, `/kaggle/input`, `/kaggle/working`, and the
current directory for a folder that directly contains both a `train/` and a
`student_test/` subfolder, so it works unchanged on Kaggle or Colab. If it can't
find your data, set `DATA_ROOT` manually in this cell.


In [ ]:
def find_data_root(search_dirs=None) -> Path:
    search_dirs = search_dirs or ["/content", "/kaggle/input", "/kaggle/working", ".", ".."]
    for base in search_dirs:
        base = Path(base)
        if not base.exists():
            continue
        for candidate in [base, *base.glob("*"), *base.glob("*/*")]:
            if candidate.is_dir() and (candidate / "train").is_dir() and (candidate / "student_test").is_dir():
                return candidate
    raise FileNotFoundError(
        "Could not auto-locate a folder containing 'train/' and 'student_test/'. "
        "Set DATA_ROOT manually below, e.g.:\n"
        '  DATA_ROOT = Path("/kaggle/input/your-dataset-slug/Student folder")'
    )


DATA_ROOT = find_data_root()          # <-- override manually here if auto-detection fails
TRAIN_DIR = DATA_ROOT / "train"
STUDENT_TEST_DIR = DATA_ROOT / "student_test"

OUTPUT_DIR = Path("/kaggle/working/outputs") if Path("/kaggle/working").exists() else Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_FEATURES_PATH = OUTPUT_DIR / "train_features.parquet"
STUDENT_TEST_FEATURES_PATH = OUTPUT_DIR / "student_test_features.parquet"
TRAIN_DIAGNOSTICS_PATH = OUTPUT_DIR / "train_diagnostics.csv"
STUDENT_TEST_DIAGNOSTICS_PATH = OUTPUT_DIR / "student_test_diagnostics.csv"
SCALER_PATH = OUTPUT_DIR / "feature_scaler.joblib"

# Added: point-level route-alignment QA artifacts (never model features).
TRAIN_ALIGNMENT_PATH = OUTPUT_DIR / "train_alignment_diagnostics.parquet"
STUDENT_TEST_ALIGNMENT_PATH = OUTPUT_DIR / "student_test_alignment_diagnostics.parquet"

# Initial reference alignment parameters — inspect/tune on training data.
ALIGNMENT_SIGMA_M = 15.0
ALIGNMENT_MAX_DISTANCE_M = 70.0
ALIGNMENT_MAX_SPEED_MPS = 55.0
ALIGNMENT_JUMP_SLACK_M = 40.0

K_CANDIDATES = 5                # nearest edges retrieved per GPS point — tune and re-run
NOMINAL_SAMPLING_HZ = 1.0       # .track files are nominally 1 Hz

# The canonical list of MODEL-INPUT columns. Deliberately excludes trip_id,
# point_idx, candidate_edge_id and label — those are bookkeeping/target only.
FEATURE_COLUMNS = [
    "dist_to_edge_m", "dist_to_u_node_m", "dist_to_v_node_m",
    "bearing_diff_deg", "bearing_valid",
    "time_gap_prev_s", "speed_implied_mps", "irregular_gap_flag",
    "candidate_rank", "dist_minus_best_at_point_m", "num_candidates_at_point",
    "edge_length_m", "u_node_degree", "v_node_degree", "reverse_edge_exists",
]
LABEL_COLUMN = "label"

print(f"DATA_ROOT        = {DATA_ROOT}")
print(f"TRAIN_DIR         = {TRAIN_DIR}  ({len(list(TRAIN_DIR.iterdir()))} folders)")
print(f"STUDENT_TEST_DIR  = {STUDENT_TEST_DIR}  ({len(list(STUDENT_TEST_DIR.iterdir()))} folders)")
print(f"OUTPUT_DIR        = {OUTPUT_DIR}")

## 2. Geographic helpers

Distance features use a **local, per-trip tangent-plane projection** (equirectangular,
centred on that trip's own node centroid) rather than raw lon/lat degrees — accurate
to well under 0.1% at city-network scale, negligible next to GPS noise itself.
Bearing is computed separately, directly from lon/lat via the standard great-circle
initial-bearing formula, so it isn't affected by the projection's small local distortion.


In [ ]:
EARTH_RADIUS_M = 6_371_000.0


def make_local_projector(lon0: float, lat0: float):
    lat0_rad = np.radians(lat0)
    cos_lat0 = np.cos(lat0_rad)

    def project(lon, lat):
        lon = np.asarray(lon, dtype=float)
        lat = np.asarray(lat, dtype=float)
        x = np.radians(lon - lon0) * cos_lat0 * EARTH_RADIUS_M
        y = np.radians(lat - lat0) * EARTH_RADIUS_M
        return x, y

    return project


def point_to_segment_distance(px, py, ax, ay, bx, by):
    """Distance from point P to segment AB, clamped to the segment's endpoints."""
    abx, aby = bx - ax, by - ay
    seg_len_sq = abx * abx + aby * aby
    seg_len_sq = np.where(seg_len_sq == 0, 1e-12, seg_len_sq)  # guard zero-length edges
    t = ((px - ax) * abx + (py - ay) * aby) / seg_len_sq
    t_clamped = np.clip(t, 0.0, 1.0)
    closest_x = ax + t_clamped * abx
    closest_y = ay + t_clamped * aby
    return np.hypot(px - closest_x, py - closest_y)


def initial_bearing_deg(lon1, lat1, lon2, lat2):
    """Great-circle initial bearing in degrees [0, 360); NaN if points coincide."""
    lon1, lat1 = np.radians(lon1), np.radians(lat1)
    lon2, lat2 = np.radians(lon2), np.radians(lat2)
    dlon = lon2 - lon1
    x = np.sin(dlon) * np.cos(lat2)
    y = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(dlon)
    coincident = (np.abs(dlon) < 1e-12) & (np.abs(lat2 - lat1) < 1e-12)
    bearing = np.degrees(np.arctan2(x, y))
    bearing = np.where(bearing < 0, bearing + 360.0, bearing)
    return np.where(coincident, np.nan, bearing)


def angular_difference_deg(a, b):
    """Smallest absolute angular difference between two bearings, in [0, 180]."""
    diff = np.abs(np.asarray(a, dtype=float) - np.asarray(b, dtype=float)) % 360.0
    return np.minimum(diff, 360.0 - diff)


def haversine_m(lon1, lat1, lon2, lat2):
    """Great-circle distance in metres — used for the speed feature only."""
    lon1, lat1, lon2, lat2 = map(np.radians, (lon1, lat1, lon2, lat2))
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return 2 * EARTH_RADIUS_M * np.arcsin(np.sqrt(np.clip(a, 0, 1)))


## 3. Raw file loading

Loads exactly one trip at a time. Node/edge IDs are **trip-local** (node 5 in trip A
means nothing in trip B), so nothing here ever opens two trips' graphs together —
that boundary is what keeps IDs from being compared across trips later.


In [ ]:
@dataclass
class TripData:
    trip_id: str
    nodes: np.ndarray            # (N, 2) float: lon, lat
    arcs: np.ndarray             # (E, 2) int:   u, v  (0-based row index into nodes)
    track: np.ndarray            # (T, 3) float: lon, lat, timestamp_s
    route: Optional[np.ndarray]  # (R,) int: edge_id (row index into arcs), or None


def load_trip(trip_dir: Path, trip_id: str) -> TripData:
    nodes = np.atleast_2d(np.loadtxt(trip_dir / f"{trip_id}.nodes", dtype=float))
    arcs = np.atleast_2d(np.loadtxt(trip_dir / f"{trip_id}.arcs", dtype=int))
    track = np.atleast_2d(np.loadtxt(trip_dir / f"{trip_id}.track", dtype=float))
    route_path = trip_dir / f"{trip_id}.route"
    route = None
    if route_path.exists() and route_path.stat().st_size > 0:
        # .route is absent for the hidden evaluator trips at real inference time —
        # every function downstream must keep working with route=None.
        route = np.atleast_1d(np.loadtxt(route_path, dtype=int))
    return TripData(trip_id=trip_id, nodes=nodes, arcs=arcs, track=track, route=route)


def list_trip_ids(partition_dir: Path) -> list[str]:
    return sorted(p.name for p in partition_dir.iterdir() if p.is_dir())


## 4. Candidate generation — route-blind by construction

Notice this function's signature: there is no `route` / `true_edges` parameter
anywhere. Candidate generation is pure geometry over `(.nodes, .arcs, .track)` —
exactly what will be available at real inference time on the hidden evaluation trips.

Retrieval is **two-stage**: a KD-tree over edge midpoints finds a wide pool of
candidates fast, then that pool is re-ranked by the *true* point-to-segment distance
and only the top `k` are kept. A midpoint-only KD-tree can miss the genuinely closest
edge for long segments whose midpoint sits far from the query point — this fixes that.


In [ ]:
def generate_candidates(trip: TripData, k: int, overfetch_factor: int = 4, min_pool: int = 20):
    nodes, arcs, track = trip.nodes, trip.arcs, trip.track

    # Project using THIS trip's own centroid — never another trip's.
    lon0, lat0 = nodes[:, 0].mean(), nodes[:, 1].mean()
    project = make_local_projector(lon0, lat0)

    node_x, node_y = project(nodes[:, 0], nodes[:, 1])
    track_x, track_y = project(track[:, 0], track[:, 1])

    u_idx, v_idx = arcs[:, 0], arcs[:, 1]
    ux, uy = node_x[u_idx], node_y[u_idx]
    vx, vy = node_x[v_idx], node_y[v_idx]
    mid_x, mid_y = (ux + vx) / 2.0, (uy + vy) / 2.0

    # Stage 1 — fast, coarse retrieval over edge midpoints.
    tree = cKDTree(np.column_stack([mid_x, mid_y]))
    pool_size = min(max(k * overfetch_factor, min_pool), len(arcs))
    query_points = np.column_stack([track_x, track_y])
    _, pool_edge_ids = tree.query(query_points, k=pool_size)
    if pool_size == 1:
        pool_edge_ids = pool_edge_ids[:, None]

    n_points = len(track)
    px_pool = np.repeat(track_x, pool_size)
    py_pool = np.repeat(track_y, pool_size)
    pool_flat = pool_edge_ids.reshape(-1)
    pool_dist = point_to_segment_distance(
        px_pool, py_pool, ux[pool_flat], uy[pool_flat], vx[pool_flat], vy[pool_flat]
    ).reshape(n_points, pool_size)

    # Stage 2 — re-rank the pool by TRUE point-to-segment distance, keep top k.
    k_eff = min(k, pool_size)
    order = np.argsort(pool_dist, axis=1)[:, :k_eff]
    candidate_edge_id = np.take_along_axis(pool_edge_ids, order, axis=1).reshape(-1)
    dist_to_edge_m = np.take_along_axis(pool_dist, order, axis=1).reshape(-1)

    point_idx = np.repeat(np.arange(n_points), k_eff)
    px = np.repeat(track_x, k_eff)
    py = np.repeat(track_y, k_eff)
    dist_to_u_node_m = np.hypot(px - ux[candidate_edge_id], py - uy[candidate_edge_id])
    dist_to_v_node_m = np.hypot(px - vx[candidate_edge_id], py - vy[candidate_edge_id])

    return {
        "point_idx": point_idx,
        "candidate_edge_id": candidate_edge_id,
        "dist_to_edge_m": dist_to_edge_m,
        "dist_to_u_node_m": dist_to_u_node_m,
        "dist_to_v_node_m": dist_to_v_node_m,
        "_node_x": node_x, "_node_y": node_y,
        "_track_x": track_x, "_track_y": track_y,
    }


## 5. Feature engineering

Still route-blind. This is the function that will be reused unchanged inside the
FastAPI inference service later, so no raw `node_id`/`edge_id` value is ever used
AS a feature — only derived geometric / kinematic / topological quantities that mean
the same thing across different trips' graphs.


In [ ]:
def _vehicle_heading_deg(track):
    """Heading of travel at each point, from consecutive raw GPS fixes.
    First point has no predecessor -> NaN (flagged via bearing_valid, not guessed)."""
    n = len(track)
    heading = np.full(n, np.nan)
    if n > 1:
        heading[1:] = initial_bearing_deg(track[:-1, 0], track[:-1, 1], track[1:, 0], track[1:, 1])
    return heading


def _time_gaps_s(track):
    n = len(track)
    gap = np.full(n, np.nan)
    if n > 1:
        gap[1:] = track[1:, 2] - track[:-1, 2]
    return gap


def _implied_speed_mps(track):
    n = len(track)
    speed = np.full(n, np.nan)
    if n > 1:
        dist = haversine_m(track[:-1, 0], track[:-1, 1], track[1:, 0], track[1:, 1])
        gap = track[1:, 2] - track[:-1, 2]
        with np.errstate(divide="ignore", invalid="ignore"):
            speed[1:] = np.where(gap > 0, dist / gap, np.nan)
    return speed


def _node_degrees(arcs, n_nodes):
    """How many road segments meet at each node — structural, not an identifier."""
    degree = np.zeros(n_nodes, dtype=int)
    np.add.at(degree, arcs[:, 0], 1)
    np.add.at(degree, arcs[:, 1], 1)
    return degree


def _reverse_edge_lookup(arcs):
    """For every edge (u, v), does the reverse (v, u) also exist in this trip's graph?"""
    edge_set = set(map(tuple, arcs.tolist()))
    return np.array([(v, u) in edge_set for u, v in arcs], dtype=bool)


def build_feature_table(trip: TripData, cand: dict, nominal_sampling_hz: float) -> pd.DataFrame:
    track, arcs, n_nodes = trip.track, trip.arcs, len(trip.nodes)
    point_idx = cand["point_idx"]
    candidate_edge_id = cand["candidate_edge_id"]
    node_x, node_y = cand["_node_x"], cand["_node_y"]

    vehicle_heading = _vehicle_heading_deg(track)
    time_gap = _time_gaps_s(track)
    speed = _implied_speed_mps(track)

    df = pd.DataFrame({
        "point_idx": point_idx,
        "candidate_edge_id": candidate_edge_id,
        "timestamp": track[point_idx, 2],
        "dist_to_edge_m": cand["dist_to_edge_m"],
        "dist_to_u_node_m": cand["dist_to_u_node_m"],
        "dist_to_v_node_m": cand["dist_to_v_node_m"],
    })

    # --- directional ---
    u_idx = arcs[candidate_edge_id, 0]
    v_idx = arcs[candidate_edge_id, 1]
    edge_heading = initial_bearing_deg(
        trip.nodes[u_idx, 0], trip.nodes[u_idx, 1], trip.nodes[v_idx, 0], trip.nodes[v_idx, 1]
    )
    veh_heading_per_row = vehicle_heading[point_idx]
    bearing_diff = angular_difference_deg(veh_heading_per_row, edge_heading)
    df["bearing_valid"] = (~np.isnan(veh_heading_per_row)).astype(int)
    df["bearing_diff_deg"] = np.where(df["bearing_valid"] == 1, bearing_diff, np.nan)

    # --- kinematic / temporal ---
    df["time_gap_prev_s"] = time_gap[point_idx]
    df["speed_implied_mps"] = speed[point_idx]
    with np.errstate(invalid="ignore"):
        df["irregular_gap_flag"] = (
            np.abs(df["time_gap_prev_s"] - (1.0 / nominal_sampling_hz)) > 0.25
        ).astype(float)
    df.loc[df["time_gap_prev_s"].isna(), "irregular_gap_flag"] = np.nan

    # --- relative / within-point ranking ---
    df["candidate_rank"] = (
        df.groupby("point_idx")["dist_to_edge_m"].rank(method="first").astype(int) - 1
    )
    df["dist_minus_best_at_point_m"] = df["dist_to_edge_m"] - df.groupby("point_idx")[
        "dist_to_edge_m"
    ].transform("min")
    df["num_candidates_at_point"] = df.groupby("point_idx")["candidate_edge_id"].transform("count")

    # --- local topology (structural, generalises across trips) ---
    degree = _node_degrees(arcs, n_nodes)
    reverse_exists = _reverse_edge_lookup(arcs)
    edge_length = np.hypot(node_x[v_idx] - node_x[u_idx], node_y[v_idx] - node_y[u_idx])

    df["edge_length_m"] = edge_length
    df["u_node_degree"] = degree[u_idx]
    df["v_node_degree"] = degree[v_idx]
    df["reverse_edge_exists"] = reverse_exists[candidate_edge_id].astype(int)

    return df


In [ ]:
def align_gps_to_route(
    trip: TripData,
    sigma_m=ALIGNMENT_SIGMA_M,
    max_distance_m=ALIGNMENT_MAX_DISTANCE_M,
    max_speed_mps=ALIGNMENT_MAX_SPEED_MPS,
    jump_slack_m=ALIGNMENT_JUMP_SLACK_M,
):
    """Estimate a monotonic GPS-to-ordered-route position for labeling only.

    Runs ONLY when the .route reference is present; not used during inference.
    Alignment is estimated/pseudo-ground truth, not independent pointwise truth.
    The complete reference route is used, NOT the five generated candidates.
    """
    if trip.route is None:
        raise ValueError("A reference .route is required for training alignment")
    if sigma_m <= 0 or max_distance_m < 0 or max_speed_mps <= 0:
        raise ValueError("Invalid alignment parameter")

    route = np.asarray(trip.route, dtype=np.int64).reshape(-1)
    track = trip.track
    T, R = len(track), len(route)
    if T == 0 or R == 0:
        raise ValueError(f"{trip.trip_id}: empty GPS track or reference route")
    if np.any(route < 0) or np.any(route >= len(trip.arcs)):
        raise ValueError(f"{trip.trip_id}: reference edge ID outside arc table")

    # Position on the ordered route, NOT a de-duplicated edge ID.
    # This also distinguishes repeated visits to the same edge.
    route_arcs = trip.arcs[route]

    lon0 = float(trip.nodes[:, 0].mean())
    lat0 = float(trip.nodes[:, 1].mean())
    project = make_local_projector(lon0, lat0)
    node_x, node_y = project(trip.nodes[:, 0], trip.nodes[:, 1])
    px, py = project(track[:, 0], track[:, 1])

    ax = node_x[route_arcs[:, 0]]
    ay = node_y[route_arcs[:, 0]]
    bx = node_x[route_arcs[:, 1]]
    by = node_y[route_arcs[:, 1]]

    lengths = np.hypot(bx - ax, by - ay)
    route_start = np.r_[0.0, np.cumsum(lengths[:-1])]
    route_center = route_start + lengths / 2.0
    route_total = float(lengths.sum())

    heading = _vehicle_heading_deg(track)
    speed = _implied_speed_mps(track)
    gaps = _time_gaps_s(track)
    road_heading = initial_bearing_deg(
        trip.nodes[route_arcs[:, 0], 0],
        trip.nodes[route_arcs[:, 0], 1],
        trip.nodes[route_arcs[:, 1], 0],
        trip.nodes[route_arcs[:, 1], 1],
    )

    # O(T*R) time and O(T*R) backpointer memory; vectorized over R.
    # A state is a POSITION in the ordered route, including repeated edges.
    back = np.zeros((T, R), dtype=np.int32)
    ids = np.arange(R, dtype=np.int32)
    previous = None

    for i in range(T):
        if np.isfinite(px[i]) and np.isfinite(py[i]):
            distance = point_to_segment_distance(
                px[i], py[i], ax, ay, bx, by
            )
            # Huber-style geometric penalty; robust against noisy GPS fixes.
            z = np.minimum(distance / sigma_m, 8.0)
            emission = np.where(z <= 2.0, 0.5 * z**2, 2.0 * (z - 1.0))
        else:
            emission = np.zeros(R, dtype=float)

        if (
            np.isfinite(heading[i])
            and np.isfinite(speed[i])
            and speed[i] >= 2.0
            and np.isfinite(gaps[i])
            and 0 < gaps[i] <= 10.0
        ):
            heading_difference = angular_difference_deg(heading[i], road_heading)
            emission += np.nan_to_num(
                0.5 * (heading_difference / 90.0)**2, nan=0.0
            )

        if i == 0:
            # Soft preference for starting near the beginning of the route.
            previous = emission + 0.008 * route_center
            continue

        dt = gaps[i]
        if not np.isfinite(dt) or dt <= 0:
            dt = 1.0

        # Forward-only DP. Minimizing prev[k] + w*(pos[j]-pos[k]), k<=j,
        # can be computed with a prefix minimum rather than O(R^2) pairs.
        weight = 0.008 / max(dt, 1.0)
        adjusted = previous - weight * route_center
        prefix_min = np.minimum.accumulate(adjusted)
        is_new_best = adjusted < np.r_[np.inf, prefix_min[:-1]]
        predecessor = np.maximum.accumulate(np.where(is_new_best, ids, 0))
        back[i] = predecessor
        previous = emission + weight * route_center + prefix_min

    # Soft preference to finish near the end of the route.
    final_cost = previous + 0.008 * (route_total - route_center)
    j = int(np.argmin(final_cost))
    positions = np.empty(T, dtype=np.int32)
    for i in range(T - 1, -1, -1):
        positions[i] = j
        if i > 0:
            j = int(back[i, j])

    selected_edges = route[positions]
    distance_to_match = point_to_segment_distance(
        px, py, ax[positions], ay[positions], bx[positions], by[positions]
    )

    # Project each GPS point onto its assigned segment, in route-distance units.
    dx, dy = bx[positions] - ax[positions], by[positions] - ay[positions]
    denominator = np.maximum(dx * dx + dy * dy, 1e-12)
    fraction = np.clip(
        ((px - ax[positions]) * dx + (py - ay[positions]) * dy)
        / denominator, 0.0, 1.0
    )
    along_route_m = route_start[positions] + fraction * lengths[positions]

    # Quality flags: unreliable observations are not given supervised labels.
    status = np.full(T, "accepted", dtype=object)
    invalid_gps = (
        ~np.isfinite(px) | ~np.isfinite(py) | ~np.isfinite(track[:, 2])
    )
    status[invalid_gps] = "invalid_gps"
    status[~invalid_gps & (distance_to_match > max_distance_m)] = (
        "uncertain_distance"
    )

    if T > 1:
        dt = np.diff(track[:, 2])
        route_progress = np.diff(along_route_m)
        bad_time = ~np.isfinite(dt) | (dt <= 0)
        status[np.flatnonzero(bad_time) + 1] = "invalid_timestamp"
        jump = (
            ~bad_time
            & (route_progress > max_speed_mps * dt + jump_slack_m)
        )
        status[np.flatnonzero(jump) + 1] = "uncertain_transition"

        # Gaps between consecutive arcs in the known reference sequence.
        breaks = np.zeros(R, dtype=np.int32)
        if R > 1:
            breaks[1:] = (
                route_arcs[:-1, 1] != route_arcs[1:, 0]
            ).astype(np.int32)
        cumulative_breaks = np.cumsum(breaks)
        crosses_gap = (
            cumulative_breaks[positions[1:]]
            > cumulative_breaks[positions[:-1]]
        )
        status[np.flatnonzero(crosses_gap) + 1] = "uncertain_route_gap"
        # Preserve invalid GPS and invalid timestamp flags as stronger reasons.
        status[invalid_gps] = "invalid_gps"
        status[np.flatnonzero(bad_time) + 1] = "invalid_timestamp"

    return pd.DataFrame({
        "point_idx": np.arange(T, dtype=np.int32),
        "aligned_route_pos": positions,
        "aligned_edge_id": selected_edges,
        "alignment_distance_m": distance_to_match,
        "aligned_route_distance_m": along_route_m,
        "alignment_status": status,
    })

## 6. Reference-route alignment and point-level labeling (training only)

The new `align_gps_to_route()` function estimates a forward-moving assignment
from GPS observations to **positions in the ordered `.route`**. It uses route
geometry and temporal movement information independently of the retrieved top-K
candidate set. Repeated occurrences of the same road remain distinct positions.

The label is then 1 only for the candidate matching the estimated road **at that
specific GPS observation**. Observations with uncertain alignment or whose
estimated aligned road is absent from the candidate set retain `NaN` labels and
are excluded from ordinary supervised fitting. Alignment diagnostics remain
separate from model inputs. Candidate miss rate is **estimated** from the
pseudo-alignment, not a verified point-level ground truth metric.

In [ ]:
def attach_point_labels(df: pd.DataFrame, alignment: pd.DataFrame):
    """Attach estimated per-observation 0/1 labels; return QA separately.

    A point is labeled only if its alignment is accepted AND its aligned edge
    is among the generated candidates. Otherwise the entire group's label
    remains NaN (not five false negative examples).
    """
    df = df.copy()
    df = df.merge(
        alignment[["point_idx", "aligned_edge_id", "alignment_status"]],
        on="point_idx", how="left", validate="many_to_one"
    )
    matches = df["candidate_edge_id"] == df["aligned_edge_id"]
    hits = matches.groupby(df["point_idx"]).any()

    alignment = alignment.copy()
    alignment["candidate_hit"] = (
        alignment["point_idx"].map(hits).fillna(False).astype(bool)
    )
    alignment["usable_for_training"] = (
        alignment["alignment_status"].eq("accepted")
        & alignment["candidate_hit"]
    )
    usable = df["point_idx"].map(
        alignment.set_index("point_idx")["usable_for_training"]
    ).fillna(False)
    df["label"] = np.where(usable, matches.astype(float), np.nan)

    # Reference-derived data must never become input features or inference fields.
    df.drop(columns=["aligned_edge_id", "alignment_status"], inplace=True)
    return df, alignment


def compute_trip_diagnostics(df: pd.DataFrame, trip: TripData, alignment=None):
    n_points = int(df["point_idx"].nunique())
    n_rows = int(len(df))
    diag = {
        "trip_id": trip.trip_id,
        "n_track_points": n_points,
        "n_candidate_rows": n_rows,
        "avg_candidates_per_point": n_rows / n_points if n_points else np.nan,
    }
    if alignment is None:
        return diag

    accepted = alignment["alignment_status"].eq("accepted")
    usable = alignment["usable_for_training"]
    misses = accepted & ~alignment["candidate_hit"]
    labeled = df["label"].dropna()
    diag.update({
        "n_route_edges": int(len(trip.route)),
        "n_accepted_alignments": int(accepted.sum()),
        "n_uncertain_alignments": int((~accepted).sum()),
        "alignment_uncertain_rate": float((~accepted).mean()),
        "n_candidate_misses_estimated": int(misses.sum()),
        "candidate_miss_rate": (
            float((~alignment.loc[accepted, "candidate_hit"]).mean())
            if accepted.any() else np.nan
        ),
        "n_usable_labeled_points": int(usable.sum()),
        "n_positive_rows": int(df["label"].eq(1).sum()),
        "positive_rate_on_labeled_rows": (
            float(labeled.mean()) if len(labeled) else np.nan
        ),
    })
    return diag

## 7. Orchestration — per trip, then per partition

`build_trip_table()` now returns **route-blind candidate features only**, and is
safe to use in the FastAPI inference service when `.route` is unavailable.
`build_partition_table()` processes each trip independently, adds alignment-based
pseudo-labels only when `.route` exists, and returns features, trip-level QA, and
separate point-level alignment diagnostics.

In [ ]:
def build_trip_table(trip: TripData, k: int = K_CANDIDATES) -> pd.DataFrame:
    """Route-blind candidate/feature generation; safe for live inference."""
    cand = generate_candidates(trip, k=k)
    df = build_feature_table(trip, cand, nominal_sampling_hz=NOMINAL_SAMPLING_HZ)
    df.insert(0, "trip_id", trip.trip_id)
    return df


def build_partition_table(partition_dir: Path, k: int = K_CANDIDATES):
    trip_ids = list_trip_ids(partition_dir)
    trip_frames, diagnostics, alignment_frames = [], [], []
    for trip_id in trip_ids:
        trip = load_trip(partition_dir / trip_id, trip_id)
        trip_df = build_trip_table(trip, k=k)
        alignment = None
        if trip.route is not None:
            alignment = align_gps_to_route(trip)
            trip_df, alignment = attach_point_labels(trip_df, alignment)
            alignment.insert(0, "trip_id", trip_id)
            alignment_frames.append(alignment)
        trip_frames.append(trip_df)
        diagnostics.append(compute_trip_diagnostics(trip_df, trip, alignment))

    features_df = pd.concat(trip_frames, ignore_index=True)
    diagnostics_df = pd.DataFrame(diagnostics)
    alignment_df = (
        pd.concat(alignment_frames, ignore_index=True)
        if alignment_frames else pd.DataFrame()
    )
    return features_df, diagnostics_df, alignment_df

## 8. Leakage checks, group-aware CV, train-only preprocessing

Three distinct leakage risks, each guarded explicitly rather than assumed away:

1. No trip's rows may span two partitions &rarr; `assert_disjoint_trip_ids`
2. CV folds must not split one trip across folds &rarr; `group_aware_cv_splits`
   (wraps `GroupKFold` on `trip_id`, with an assertion that no fold overlaps —
   a plain row-level `KFold` would leak a trip's own points across train/val and
   inflate your CV scores)
3. Any fitted preprocessing must be fit on train only &rarr;
   `fit_preprocessor_on_train_only` / `apply_preprocessor`


In [ ]:
def assert_disjoint_trip_ids(df_a, df_b, name_a, name_b):
    ids_a, ids_b = set(df_a["trip_id"].unique()), set(df_b["trip_id"].unique())
    overlap = ids_a & ids_b
    assert not overlap, f"Leakage: trip_id(s) in both {name_a} and {name_b}: {sorted(overlap)[:10]}"


def assert_no_label_leaking_columns(df, feature_columns):
    banned = {"label", "trip_id", "candidate_edge_id", "point_idx"}
    leaked = banned & set(feature_columns)
    assert not leaked, f"Leakage: banned column(s) in feature list: {leaked}"


def group_aware_cv_splits(df, n_splits=5):
    """Yields (train_idx, val_idx) where every row from a given trip_id is
    entirely on one side of the split — the only correct way to CV this table."""
    gkf = GroupKFold(n_splits=n_splits)
    groups = df["trip_id"].values
    for train_idx, val_idx in gkf.split(df, groups=groups):
        assert not (set(groups[train_idx]) & set(groups[val_idx])), "GroupKFold overlap!"
        yield train_idx, val_idx


def fit_preprocessor_on_train_only(train_df, feature_columns):
    """Impute (median) then scale, fit on TRAIN ONLY. Save and reuse (transform-only)
    on student_test and, later, on inference input — never re-fit."""
    assert_no_label_leaking_columns(train_df, feature_columns)
    pipeline = Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())])
    pipeline.fit(train_df[feature_columns].to_numpy(dtype=float))
    return pipeline


def apply_preprocessor(df, feature_columns, pipeline):
    return pipeline.transform(df[feature_columns].to_numpy(dtype=float))



def assert_point_label_invariants(df, alignment):
    """Check label consistency, NOT whether estimated alignment is correct."""
    keys = ["trip_id", "point_idx"]
    positives = df.groupby(keys)["label"].sum(min_count=1)
    usable = alignment.set_index(keys)["usable_for_training"]
    positives = positives.reindex(usable.index)

    assert positives[usable].eq(1).all(), (
        "Each usable GPS point must have exactly one positive candidate"
    )
    assert positives[~usable].isna().all(), (
        "Unusable GPS points must have entirely missing labels"
    )
    assert df["label"].dropna().isin([0, 1]).all(), (
        "Nonmissing point-level labels must be 0 or 1"
    )
    forbidden = {
        "aligned_edge_id", "aligned_route_pos", "candidate_hit",
        "alignment_status", "usable_for_training",
    }
    assert not (forbidden & set(df.columns)), (
        "Reference alignment data leaked into candidate feature table"
    )
    print("Pointwise label structure validated.")

## 9. Run the full build

This cell builds both feature tables, validates estimated point-level label
structure, fits the original preprocessing pipeline using labeled training rows
only, and exports alignment diagnostics separately. The new route-alignment DP is
more expensive than the original membership test; runtime depends on GPS and
reference-route lengths. Check alignment quality before using the labels.

In [ ]:
print("[1/7] Building TRAIN partition ...")
t0 = time.time()
train_df, train_diag, train_alignment = build_partition_table(TRAIN_DIR, k=K_CANDIDATES)
print(f"      {len(train_df):,} rows from {train_df['trip_id'].nunique()} trips in {time.time()-t0:.1f}s")

print("[2/7] Building STUDENT_TEST partition ...")
t0 = time.time()
test_df, test_diag, test_alignment = build_partition_table(STUDENT_TEST_DIR, k=K_CANDIDATES)
print(f"      {len(test_df):,} rows from {test_df['trip_id'].nunique()} trips in {time.time()-t0:.1f}s")

print("[3/7] Trip-level leakage and group-aware CV checks ...")
assert_disjoint_trip_ids(train_df, test_df, "train", "student_test")
for _ in group_aware_cv_splits(train_df, n_splits=5):
    pass
print("      Trip partitions and 5-fold group CV verified.")

print("[4/7] Point-level label consistency checks ...")
assert_point_label_invariants(train_df, train_alignment)
assert_point_label_invariants(test_df, test_alignment)
assert len(FEATURE_COLUMNS) == 15
assert_no_label_leaking_columns(train_df, FEATURE_COLUMNS)

print("[5/7] Fitting preprocessing pipeline on TRAIN labeled rows only ...")
train_labeled = train_df.loc[train_df["label"].notna()].copy()
if train_labeled.empty:
    raise ValueError("No usable training observations after route alignment")
preprocessor = fit_preprocessor_on_train_only(train_labeled, FEATURE_COLUMNS)
joblib.dump(preprocessor, SCALER_PATH)
apply_preprocessor(test_df, FEATURE_COLUMNS, preprocessor)

print("[6/7] Writing Parquet, diagnostics, and scaler ...")
train_df.to_parquet(TRAIN_FEATURES_PATH, index=False)
test_df.to_parquet(STUDENT_TEST_FEATURES_PATH, index=False)
train_diag.to_csv(TRAIN_DIAGNOSTICS_PATH, index=False)
test_diag.to_csv(STUDENT_TEST_DIAGNOSTICS_PATH, index=False)
train_alignment.to_parquet(TRAIN_ALIGNMENT_PATH, index=False)
test_alignment.to_parquet(STUDENT_TEST_ALIGNMENT_PATH, index=False)

print("[7/7] Summary")
print(f"      train_features.parquet           : {train_df.shape}")
print(f"      student_test_features.parquet    : {test_df.shape}")
print(f"      train labeled observations       : {int(train_alignment['usable_for_training'].sum()):,}")
print(f"      student_test labeled observations: {int(test_alignment['usable_for_training'].sum()):,}")
print(f"      train positive rate (labeled)    : {train_df['label'].mean():.3f}")
print(f"      student test positive (labeled)  : {test_df['label'].mean():.3f}")
print(f"      train candidate miss (estimated) : {train_diag['candidate_miss_rate'].mean():.4f}")
print(f"      test candidate miss (estimated)  : {test_diag['candidate_miss_rate'].mean():.4f}")
print("Done. Files are in:", OUTPUT_DIR.resolve())

## 10. Quick sanity check

View candidate features and estimated alignment diagnostics. The candidate miss
rate is now evaluated against the estimated road at each GPS observation,
not against membership anywhere in the trip. Validate uncertain alignments
and visualize selected trip reconstructions before treating these as labels.

In [ ]:
display(train_df.head(8))
print()
print("Per-trip diagnostics (train), sorted by estimated candidate_miss_rate:")
display(train_diag.sort_values("candidate_miss_rate", ascending=False).head(10))
print()
print(train_diag[["n_track_points", "n_usable_labeled_points", "positive_rate_on_labeled_rows", "candidate_miss_rate"]].describe())

## 11. (Colab only) download the outputs

On Kaggle, files in `/kaggle/working` can be saved with the notebook version.
On Colab, the runtime is ephemeral, so download the files before disconnecting:
```python
from google.colab import files
for p in [TRAIN_FEATURES_PATH, STUDENT_TEST_FEATURES_PATH,
          TRAIN_DIAGNOSTICS_PATH, STUDENT_TEST_DIAGNOSTICS_PATH,
          TRAIN_ALIGNMENT_PATH, STUDENT_TEST_ALIGNMENT_PATH, SCALER_PATH]:
    files.download(str(p))
```